In [2]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Binary Mixture Optimization for ORC - Interactive Tutorial\n",
    "\n",
    "This notebook demonstrates:\n",
    "1. Geometric projection in one-hot space\n",
    "2. Property calculation with REFPROP vs mixing rules\n",
    "3. REFPROP mixture evaluation\n",
    "4. Running small-scale BO experiments\n",
    "5. Analyzing and visualizing results\n",
    "\n",
    "**Prerequisites:**\n",
    "- All Python files in same directory\n",
    "- REFPROP installed and accessible\n",
    "- Joback_Refrigerants.csv available"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Setup & Imports"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Imports\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import CoolProp.CoolProp as CP\n",
    "\n",
    "from mixture_geometry import (\n",
    "    snap_to_mixture,\n",
    "    compute_mixture_properties,\n",
    "    make_refprop_mixture_string,\n",
    "    format_mixture_name,\n",
    ")\n",
    "\n",
    "sns.set_style('whitegrid')\n",
    "%matplotlib inline\n",
    "\n",
    "print(\"✓ Imports successful!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 1: Understanding the Geometric Projection\n",
    "\n",
    "We'll use a simple 3-fluid example to visualize how continuous suggestions map to pure fluids or mixtures."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Setup toy 3-fluid system\n",
    "dtype = torch.double\n",
    "device = \"cpu\"\n",
    "tkwargs = {\"device\": device, \"dtype\": dtype}\n",
    "\n",
    "# Three fluids: R32, R125, R134a\n",
    "fluid_names = [\"R32\", \"R125\", \"R134a\"]\n",
    "t_dim = 3\n",
    "onehot = torch.eye(t_dim, **tkwargs)\n",
    "evaluated_edges = set()\n",
    "\n",
    "# Vertices in 3D\n",
    "print(\"One-hot vertices:\")\n",
    "for i, name in enumerate(fluid_names):\n",
    "    print(f\"  {name}: {onehot[i].numpy()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test suggestions\n",
    "test_cases = [\n",
    "    (\"Near R32\", torch.tensor([0.95, 0.03, 0.02], **tkwargs)),\n",
    "    (\"50:50 R32-R125\", torch.tensor([0.5, 0.5, 0.0], **tkwargs)),\n",
    "    (\"70:30 R32-R125\", torch.tensor([0.7, 0.3, 0.0], **tkwargs)),\n",
    "    (\"Interior point\", torch.tensor([0.4, 0.4, 0.2], **tkwargs)),\n",
    "]\n",
    "\n",
    "results = []\n",
    "for desc, x_suggest in test_cases:\n",
    "    j1, j2, x1 = snap_to_mixture(x_suggest, onehot, evaluated_edges)\n",
    "    \n",
    "    if j2 is None:\n",
    "        result = f\"Pure {fluid_names[j1]}\"\n",
    "    else:\n",
    "        result = f\"{fluid_names[j1]}[{x1:.2f}] & {fluid_names[j2]}[{1-x1:.2f}]\"\n",
    "    \n",
    "    results.append({\n",
    "        'Description': desc,\n",
    "        'Suggestion': str(x_suggest.numpy()),\n",
    "        'Result': result,\n",
    "    })\n",
    "\n",
    "df_snap = pd.DataFrame(results)\n",
    "print(\"\\nGeometric Projection Examples:\")\n",
    "print(df_snap.to_string(index=False))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize in 2D projection (first two components)\n",
    "fig, ax = plt.subplots(figsize=(8, 8))\n",
    "\n",
    "# Vertices\n",
    "vertices_2d = onehot[:, :2].numpy()\n",
    "ax.scatter(vertices_2d[:, 0], vertices_2d[:, 1], \n",
    "           s=300, c='red', marker='o', edgecolors='k', linewidths=2,\n",
    "           label='Pure fluids (vertices)', zorder=10)\n",
    "\n",
    "# Label vertices\n",
    "for i, name in enumerate(fluid_names):\n",
    "    ax.annotate(name, vertices_2d[i], xytext=(5, 5), \n",
    "                textcoords='offset points', fontsize=12, fontweight='bold')\n",
    "\n",
    "# Edges\n",
    "for i in range(t_dim):\n",
    "    for j in range(i+1, t_dim):\n",
    "        ax.plot([vertices_2d[i, 0], vertices_2d[j, 0]], \n",
    "                [vertices_2d[i, 1], vertices_2d[j, 1]], \n",
    "                'k--', alpha=0.3, linewidth=1)\n",
    "\n",
    "# Test suggestions\n",
    "suggestions_2d = np.array([x.numpy()[:2] for _, x in test_cases])\n",
    "ax.scatter(suggestions_2d[:, 0], suggestions_2d[:, 1],\n",
    "           s=150, c='blue', marker='x', linewidths=3,\n",
    "           label='Suggestions', zorder=5)\n",
    "\n",
    "# Annotate suggestions\n",
    "for i, (desc, _) in enumerate(test_cases):\n",
    "    ax.annotate(f\"{i+1}\", suggestions_2d[i], xytext=(5, -15),\n",
    "                textcoords='offset points', fontsize=10, color='blue')\n",
    "\n",
    "ax.set_xlabel('Component 1 (R32)', fontsize=12)\n",
    "ax.set_ylabel('Component 2 (R125)', fontsize=12)\n",
    "ax.set_title('Geometric Projection in One-Hot Space\\n(2D projection)', \n",
    "             fontsize=14, fontweight='bold')\n",
    "ax.legend(loc='upper right', fontsize=11)\n",
    "ax.grid(True, alpha=0.3)\n",
    "ax.set_xlim(-0.1, 1.1)\n",
    "ax.set_ylim(-0.1, 1.1)\n",
    "ax.set_aspect('equal')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 2: Property Calculations - REFPROP vs Mixing Rules\n",
    "\n",
    "Compare different methods for computing mixture critical properties."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compare REFPROP vs mixing rules\n",
    "fluid1, fluid2 = \"R32\", \"R125\"\n",
    "property_cache = {}\n",
    "\n",
    "# Get pure properties\n",
    "Tc1 = CP.PropsSI(\"Tcrit\", fluid1)\n",
    "Pc1 = CP.PropsSI(\"Pcrit\", fluid1)\n",
    "Tc2 = CP.PropsSI(\"Tcrit\", fluid2)\n",
    "Pc2 = CP.PropsSI(\"Pcrit\", fluid2)\n",
    "\n",
    "print(f\"Pure fluid properties:\")\n",
    "print(f\"  {fluid1}: Tc={Tc1:.1f}K, Pc={Pc1/1e6:.2f}MPa\")\n",
    "print(f\"  {fluid2}: Tc={Tc2:.1f}K, Pc={Pc2/1e6:.2f}MPa\")\n",
    "print()\n",
    "\n",
    "# Sweep compositions\n",
    "x1_values = np.linspace(0, 1, 21)\n",
    "results_comparison = []\n",
    "\n",
    "for x1 in x1_values:\n",
    "    # Method 1: Our implementation (tries REFPROP first, then inverse-sum)\n",
    "    Tc_impl, Pc_impl = compute_mixture_properties(fluid1, fluid2, x1, {}, property_cache)\n",
    "    \n",
    "    # Method 2: Kay's rule (for comparison)\n",
    "    if x1 == 0:\n",
    "        Tc_kay, Pc_kay = Tc2, Pc2\n",
    "    elif x1 == 1:\n",
    "        Tc_kay, Pc_kay = Tc1, Pc1\n",
    "    else:\n",
    "        Tc_kay = x1 * Tc1 + (1-x1) * Tc2\n",
    "        Pc_kay = x1 * Pc1 + (1-x1) * Pc2\n",
    "    \n",
    "    # Method 3: Inverse-sum (manual)\n",
    "    if x1 == 0:\n",
    "        Tc_inv, Pc_inv = Tc2, Pc2\n",
    "    elif x1 == 1:\n",
    "        Tc_inv, Pc_inv = Tc1, Pc1\n",
    "    else:\n",
    "        Tc_inv = x1 * Tc1 + (1-x1) * Tc2\n",
    "        Pc_inv = 1.0 / (x1/Pc1 + (1-x1)/Pc2)\n",
    "    \n",
    "    results_comparison.append({\n",
    "        'x1': x1,\n",
    "        'Tc_Implementation': Tc_impl,\n",
    "        'Pc_Implementation': Pc_impl,\n",
    "        'Tc_Kay': Tc_kay,\n",
    "        'Pc_Kay': Pc_kay,\n",
    "        'Tc_InverseSum': Tc_inv,\n",
    "        'Pc_InverseSum': Pc_inv,\n",
    "    })\n",
    "\n",
    "df_comp = pd.DataFrame(results_comparison)\n",
    "\n",
    "# Plot comparison\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Tc comparison\n",
    "axes[0].plot(df_comp['x1'], df_comp['Tc_Implementation'], 'o-', \n",
    "             label=\"Implementation (REFPROP or Inverse)\", linewidth=2, markersize=6)\n",
    "axes[0].plot(df_comp['x1'], df_comp['Tc_Kay'], 's--', \n",
    "             label=\"Kay's Rule\", linewidth=2, markersize=6, alpha=0.7)\n",
    "axes[0].set_xlabel(f'Mole Fraction of {fluid1}', fontsize=12)\n",
    "axes[0].set_ylabel('Critical Temperature [K]', fontsize=12)\n",
    "axes[0].set_title('Critical Temperature', fontsize=13, fontweight='bold')\n",
    "axes[0].legend(fontsize=11)\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Pc comparison\n",
    "axes[1].plot(df_comp['x1'], df_comp['Pc_Implementation']/1e6, 'o-', \n",
    "             label=\"Implementation (REFPROP or Inverse)\", linewidth=2, markersize=6)\n",
    "axes[1].plot(df_comp['x1'], df_comp['Pc_Kay']/1e6, 's--', \n",
    "             label=\"Kay's Linear\", linewidth=2, markersize=6, alpha=0.7)\n",
    "axes[1].plot(df_comp['x1'], df_comp['Pc_InverseSum']/1e6, '^:', \n",
    "             label=\"Inverse-Sum\", linewidth=2, markersize=6, alpha=0.7)\n",
    "axes[1].set_xlabel(f'Mole Fraction of {fluid1}', fontsize=12)\n",
    "axes[1].set_ylabel('Critical Pressure [MPa]', fontsize=12)\n",
    "axes[1].set_title('Critical Pressure', fontsize=13, fontweight='bold')\n",
    "axes[1].legend(fontsize=11)\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f\"\\nNote: Implementation uses REFPROP when available, otherwise inverse-sum rule.\")\n",
    "print(f\"      Kay's linear rule typically has 3-5% error for Pc.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 3: ORC Simulation for a Mixture\n",
    "\n",
    "Evaluate cycle efficiency for a specific mixture composition."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ORC parameters\n",
    "Tin_src, Tout_src = 150, 135\n",
    "mfr_src = 15\n",
    "pump_eff, turb_eff, gen_eff = 0.8, 0.65, 0.97\n",
    "hin_src = CP.PropsSI(\"H\",\"T\",Tin_src+273.15,\"P\",5e5,\"water\")\n",
    "hout_src = CP.PropsSI(\"H\",\"T\",Tout_src+273.15,\"P\",5e5,\"water\")\n",
    "\n",
    "def simple_orc_eval(wf_string, p_evap_bar, p_cond_bar):\n",
    "    \"\"\"Quick ORC evaluation (simplified - no full isentropic helper)\"\"\"\n",
    "    try:\n",
    "        p_evap = p_evap_bar * 1e5\n",
    "        p_cond = p_cond_bar * 1e5\n",
    "        \n",
    "        if p_cond >= p_evap:\n",
    "            return -1.0\n",
    "        \n",
    "        # Saturated states\n",
    "        h_g = CP.PropsSI(\"Hmass\", \"Q\", 1, \"P\", p_evap, wf_string)\n",
    "        s_g = CP.PropsSI(\"Smass\", \"Q\", 1, \"P\", p_evap, wf_string)\n",
    "        h_f = CP.PropsSI(\"Hmass\", \"Q\", 0, \"P\", p_cond, wf_string)\n",
    "        \n",
    "        # Turbine (simplified)\n",
    "        try:\n",
    "            h2s = CP.PropsSI(\"Hmass\", \"Smass\", s_g, \"P\", p_cond, wf_string)\n",
    "        except:\n",
    "            # If direct inversion fails, estimate\n",
    "            h2s = h_f + 0.5 * (h_g - h_f)\n",
    "        h2 = h_g - turb_eff * (h_g - h2s)\n",
    "        \n",
    "        # Pump\n",
    "        s_f = CP.PropsSI(\"Smass\", \"Q\", 0, \"P\", p_cond, wf_string)\n",
    "        try:\n",
    "            h1s = CP.PropsSI(\"Hmass\", \"Smass\", s_f, \"P\", p_evap, wf_string)\n",
    "        except:\n",
    "            h1s = h_f + 0.01 * (h_g - h_f)\n",
    "        h1 = h_f - (h_f - h1s) / pump_eff\n",
    "        \n",
    "        # Energy balance\n",
    "        Q_evap = mfr_src * (hin_src - hout_src)\n",
    "        if (h_g - h1) <= 0:\n",
    "            return -1.0\n",
    "        mfr_wf = Q_evap / (h_g - h1)\n",
    "        W_pump = mfr_wf * (h1 - h_f)\n",
    "        W_turb = mfr_wf * (h_g - h2)\n",
    "        eta = (gen_eff * W_turb - W_pump) / Q_evap\n",
    "        \n",
    "        if not np.isfinite(eta) or eta < 0 or eta > 0.6:\n",
    "            return -1.0\n",
    "        \n",
    "        return eta\n",
    "    except:\n",
    "        return -1.0\n",
    "\n",
    "# Test a mixture\n",
    "test_mixture = make_refprop_mixture_string(\"R32\", \"R125\", 0.7)\n",
    "print(f\"Testing mixture: {test_mixture}\\n\")\n",
    "\n",
    "# Sweep operating pressures\n",
    "p_evap_test = 20.0  # bar\n",
    "p_cond_range = np.linspace(2, 10, 20)\n",
    "\n",
    "efficiencies = [simple_orc_eval(test_mixture, p_evap_test, pc) for pc in p_cond_range]\n",
    "\n",
    "# Plot\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "ax.plot(p_cond_range, efficiencies, 'o-', linewidth=2, markersize=8, color='steelblue')\n",
    "ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)\n",
    "ax.set_xlabel('Condenser Pressure [bar]', fontsize=12)\n",
    "ax.set_ylabel('Cycle Efficiency η', fontsize=12)\n",
    "ax.set_title(f'ORC Efficiency vs Condenser Pressure\\n{test_mixture}\\n(p_evap = {p_evap_test} bar)', \n",
    "             fontsize=13, fontweight='bold')\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "valid_eff = [e for e in efficiencies if e > 0]\n",
    "if valid_eff:\n",
    "    best_idx = np.argmax(efficiencies)\n",
    "    print(f\"Best efficiency: η = {efficiencies[best_idx]:.4f} at p_cond = {p_cond_range[best_idx]:.1f} bar\")\n",
    "else:\n",
    "    print(\"No valid operating points found - check pressure bounds\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 4: Run Mini BO Experiment\n",
    "\n",
    "Run a small-scale mixture optimization and analyze results."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import subprocess\n",
    "import sys\n",
    "\n",
    "# Run 1-stage mixture BO with small budget\n",
    "print(\"Running mini mixture optimization...\\n\")\n",
    "print(\"This will take 2-5 minutes depending on SCBO convergence.\\n\")\n",
    "\n",
    "os.environ['PYTHONHASHSEED'] = '999'\n",
    "\n",
    "cmd = [\n",
    "    sys.executable, \"onestage_mixture.py\",\n",
    "    \"--csv\", \"Joback_Refrigerants.csv\",\n",
    "    \"--n-init\", \"2\",\n",
    "    \"--scbo-budget\", \"2\",\n",
    "    \"--outdir\", \"notebook_example\"\n",
    "]\n",
    "\n",
    "result = subprocess.run(cmd, capture_output=True, text=True)\n",
    "\n",
    "if result.returncode == 0:\n",
    "    print(\"✓ Optimization completed!\\n\")\n",
    "    # Print last portion of output\n",
    "    output_lines = result.stdout.split('\\n')\n",
    "    print(\"\\n\".join(output_lines[-20:]))  # Last 20 lines\n",
    "else:\n",
    "    print(\"✗ Optimization failed:\\n\")\n",
    "    print(result.stderr)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and analyze results\n",
    "results_path = \"notebook_example/seed_999/scbo_results.csv\"\n",
    "\n",
    "if os.path.exists(results_path):\n",
    "    df_results = pd.read_csv(results_path)\n",
    "    \n",
    "    print(\"\\n\" + \"=\"*80)\n",
    "    print(\"RESULTS SUMMARY\")\n",
    "    print(\"=\"*80)\n",
    "    \n",
    "    # Display key columns\n",
    "    display_cols = ['mixture', 'eta_best', 'p_evap_bar', 'p_cond_bar']\n",
    "    if 'mixture' not in df_results.columns:\n",
    "        display_cols[0] = 'fluid'\n",
    "    \n",
    "    print(df_results[display_cols].to_string(index=False))\n",
    "    \n",
    "    # Find best\n",
    "    feasible = df_results[df_results['eta_best'] >= 0]\n",
    "    if len(feasible) > 0:\n",
    "        best_idx = feasible['eta_best'].idxmax()\n",
    "        best_row = df_results.loc[best_idx]\n",
    "        \n",
    "        print(f\"\\n{'='*80}\")\n",
    "        print(f\"BEST RESULT:\")\n",
    "        print(f\"  Mixture: {best_row.get('mixture', best_row.get('fluid', 'N/A'))}\")\n",
    "        print(f\"  Efficiency: {best_row['eta_best']:.5f}\")\n",
    "        print(f\"  p_evap: {best_row['p_evap_bar']:.2f} bar\")\n",
    "        print(f\"  p_cond: {best_row['p_cond_bar']:.2f} bar\")\n",
    "        print(f\"{'='*80}\")\n",
    "    else:\n",
    "        print(\"\\n⚠ No feasible results found\")\n",
    "else:\n",
    "    print(f\"❌ Results file not found: {results_path}\")\n",
    "    print(\"\\nThe optimization may have failed. Check the output above for errors.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Part 5: Visualize Results\n",
    "\n",
    "Use the visualization tools to analyze the mini experiment."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Try importing visualization module\n",
    "try:\n",
    "    from mixture_viz import MixtureVisualizer\n",
    "    viz_available = True\n",
    "except ImportError:\n",
    "    print(\"⚠ mixture_viz.py not found in current directory\")\n",
    "    print(\"  Skipping visualization section\")\n",
    "    viz_available = False\n",
    "\n",
    "if viz_available and os.path.exists(results_path):\n",
    "    viz = MixtureVisualizer(\n",
    "        results_csv=results_path,\n",
    "        sequence_csv=\"notebook_example/seed_999/sequence.csv\"\n",
    "    )\n",
    "    \n",
    "    print(\"Generating visualizations...\\n\")\n",
    "    \n",
    "    try:\n",
    "        viz.plot_property_space()\n",
    "    except Exception as e:\n",
    "        print(f\"Property space plot failed: {e}\")\n",
    "    \n",
    "    try:\n",
    "        viz.plot_composition_histogram()\n",
    "    except Exception as e:\n",
    "        print(f\"Composition histogram failed: {e}\")\n",
    "    \n",
    "    try:\n",
    "        viz.plot_efficiency_comparison()\n",
    "    except Exception as e:\n",
    "        print(f\"Efficiency comparison failed: {e}\")\n",
    "    \n",
    "    print(\"\\n✓ Visualization complete!\")\n",
    "elif not viz_available:\n",
    "    print(\"Visualization module not available\")\n",
    "else:\n",
    "    print(\"No results to visualize. Run the optimization cell first.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary & Next Steps\n",
    "\n",
    "You've now:\n",
    "1. ✓ Seen how geometric projection works\n",
    "2. ✓ Compared REFPROP vs mixing rules\n",
    "3. ✓ Evaluated ORC performance for a mixture\n",
    "4. ✓ Run a mini BO experiment\n",
    "5. ✓ Visualized the results\n",
    "\n",
    "---\n",
    "\n",
    "### For Production Runs:\n",
    "\n",
    "```bash\n",
    "# Full 1-stage run\n",
    "python onestage_mixture.py \\\n",
    "    --csv Joback_Refrigerants.csv \\\n",
    "    --n-init 3 \\\n",
    "    --scbo-budget 10 \\\n",
    "    --outdir runs/production\n",
    "\n",
    "# Benchmark pure vs mixture\n",
    "python benchmark_pure_vs_mixture.py \\\n",
    "    --csv Joback_Refrigerants.csv \\\n",
    "    --seeds 5 \\\n",
    "    --outdir benchmark_results\n",
    "\n",
    "# Advanced 2-stage\n",
    "python twostage_mixture.py \\\n",
    "    --csv Joback_Refrigerants.csv\n",
    "```\n",
    "\n",
    "---\n",
    "\n",
    "### For Analysis:\n",
    "\n",
    "```python\n",
    "from mixture_viz import MixtureVisualizer\n",
    "viz = MixtureVisualizer(\"path/to/results.csv\")\n",
    "viz.generate_report(output_dir=\"analysis\")\n",
    "```\n",
    "\n",
    "---\n",
    "\n",
    "### Documentation:\n",
    "\n",
    "- **Quick start**: `README_MIXTURE.md`\n",
    "- **Detailed guide**: `MIXTURE_EXTENSION_GUIDE.md`\n",
    "- **Property methods**: `PROPERTY_CALCULATION_GUIDE.md`\n",
    "- **Technical reference**: `IMPLEMENTATION_SUMMARY.md`"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Clean up (optional)\n",
    "# Uncomment to remove example results\n",
    "# import shutil\n",
    "# if os.path.exists(\"notebook_example\"):\n",
    "#     shutil.rmtree(\"notebook_example\")\n",
    "#     print(\"✓ Cleaned up example results\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}

NameError: name 'null' is not defined